In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------------------------------
# 1️⃣ 准备一个极简词表（仅作演示）
# -------------------------------------------------
vocab = {"<PAD>":0, "<BOS>":1, "<EOS>":2,
         "今":3, "天":4, "天气":5, "好":6}
vocab_size = len(vocab)
idx2word = {i:w for w,i in vocab.items()}

# -------------------------------------------------
# 2️⃣ 定义模型： 嵌入 + 线性层（预测下一个词）
# -------------------------------------------------
class SimpleLM(nn.Module):
    def __init__(self, vocab_sz, embed_dim=8):
        super().__init__()
        self.embed = nn.Embedding(vocab_sz, embed_dim)   # 词嵌入
        self.fc    = nn.Linear(embed_dim, vocab_sz)      # 直接映射回词表

    def forward(self, x):
        """
        x: (batch, seq_len)  整数 id
        返回： (batch, seq_len, vocab_sz) 每个位置的概率 logits
        """
        emb = self.embed(x)               # (B, L, D)
        logits = self.fc(emb)             # (B, L, V)
        return logits

# -------------------------------------------------
# 3️⃣ 训练（演示）——用“今 天 气 好” 预测下一个词
# -------------------------------------------------
model = SimpleLM(vocab_size, embed_dim=8)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

# 训练数据（单句 + BOS/EOS）
sentence = ["<BOS>", "今", "天", "天气", "好", "<EOS>"]
ids = torch.tensor([[vocab[t] for t in sentence]])   # (1, 6)

# 输入是前 N 个词，标签是第 N+1 个词
inputs = ids[:, :-1]   # (1,5)
targets = ids[:, 1:]   # (1,5)

for epoch in range(200):
    optimizer.zero_grad()
    logits = model(inputs)                     # (1,5,V)
    loss = criterion(logits.view(-1, vocab_size), targets.view(-1))
    loss.backward()
    optimizer.step()
    if (epoch+1) % 50 == 0:
        print(f"epoch {epoch+1:3d}  loss={loss.item():.4f}")

# -------------------------------------------------
# 4️⃣ 推理——给 “今 天” 预测下一个词
# -------------------------------------------------
model.eval()
with torch.no_grad():
    demo = torch.tensor([[vocab["<BOS>"], vocab["今"], vocab["天"]]])
    out_logits = model(demo)                       # (1,3,V)
    # 只看最后一个位置的分布
    probs = F.softmax(out_logits[0, -1], dim=0)
    pred_id = torch.argmax(probs).item()
    print("预测下一个词 =", idx2word[pred_id])


epoch  50  loss=0.1687
epoch 100  loss=0.0236
epoch 150  loss=0.0104
epoch 200  loss=0.0061
预测下一个词 = 天气
